# Install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install groq python-dotenv

# Import libraries

In [ ]:
import os
from google.colab import files
import pandas as pd
import time
import json
import re  # For parsing
from groq import Groq
from dotenv import load_dotenv

# Upload file .env to Colab

In [ ]:
uploaded_file = files.upload()

# Get API Key

In [ ]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found")

# Connect with Groq

In [ ]:
groq_client = Groq(
    api_key = groq_api_key,
)

# Select LLM_as_a_Judge

In [ ]:
'''
LLM-as-a-judge models selected from Groq API provider:
['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'llama-3.1-8b-instant']

Duplicate this notebook to run per LLM-as-a-judge
Adjust judge_model accordingly
'''
judge_model = 'openai/gpt-oss-20b'

# Select models

In [ ]:
'''
List models:
groq_models = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b']
openrouter_models = ['deepseek/deepseek-chat-v3-0324:free']
'''
model_a = 'moonshotai/kimi-k2-instruct'
# Always keep model_b empty
model_b = ''

# Adjust file path to match the target environment
judge_prompts_path = 'drive/MyDrive/ViJudge/prompts/judge_prompts.jsonl'

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
name_judgement = extract_model_name(judge_model)
name_model_a = extract_model_name(model_a)
name_model_b = extract_model_name(model_b)

if model_b != '':
  input_file_path = f'drive/MyDrive/ViJudge/data/datasets_before_judgement/pair_{name_model_a}_{name_model_b}.csv'
  output_file_path = f'drive/MyDrive/ViJudge/data/gen_judgement/{name_judgement}/{name_judgement}Judge_pair_{name_model_a}_{name_model_b}.csv'
else:
  input_file_path = f'drive/MyDrive/ViJudge/data/datasets_before_judgement/single_{name_model_a}.csv'
  output_file_path = f'drive/MyDrive/ViJudge/data/gen_judgement/{name_judgement}/{name_judgement}Judge_single_{name_model_a}.csv'

print(input_file_path)
print(output_file_path)

# Load data

In [ ]:
merged_df = pd.read_csv(input_file_path)
merged_df.tail(2)

# Load prompts

In [ ]:
# Load Judge Prompts
judge_prompts = {}
with open(judge_prompts_path, 'r') as f:
    for line in f:
        prompt_data = json.loads(line)
        judge_prompts[prompt_data['name']] = prompt_data
print(f"Loaded prompts: {list(judge_prompts.keys())}")

# Generate judment

In [ ]:
# Function to Generate Judgement
def generate_judgement(client, row, judge_name):
    # Extract values, handle NaN
    question_1 = row['content_turn1'] if pd.notna(row['content_turn1']) else ''
    question_2 = row['content_turn2'] if pd.notna(row['content_turn2']) else ''
    ref_1 = row['reference_turn1'] if pd.notna(row['reference_turn1']) else ''
    ref_2 = row['reference_turn2'] if pd.notna(row['reference_turn2']) else ''
    answer_1_a = row['response_turn1_a'] if pd.notna(row['response_turn1_a']) else ''
    answer_2_a = row['response_turn2_a'] if pd.notna(row['response_turn2_a']) else ''
    answer_1_b = row['response_turn1_b'] if pd.notna(row['response_turn1_b']) else ''
    answer_2_b = row['response_turn2_b'] if pd.notna(row['response_turn2_b']) else ''

    # Is multi-turn if has question_2
    multi_turn = bool(question_2)

    # Has reference if either ref_1 or ref_2 is not empty
    has_ref = bool(ref_1 or ref_2)

    # Is pair if has response b, else single
    judge_type = 'pair' if answer_1_b or answer_2_b else 'single'

    # Define prompt name
    if multi_turn:
        prompt_name = judge_type + '_multi_turn'
    else:
        prompt_name = judge_type + '_1_turn'

    if has_ref:
        prompt_name = prompt_name + '_reference'

    if prompt_name not in judge_prompts:
        raise ValueError(f'Prompt {prompt_name} not found')

    prompt_data = judge_prompts[prompt_name]
    system_prompt = prompt_data['system_prompt']
    template = prompt_data['prompt_template']

    # Format user_prompt
    if multi_turn:
        if judge_type == 'pair':
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_a_1=answer_1_a, answer_a_2=answer_2_a,
                answer_b_1=answer_1_b, answer_b_2=answer_2_b
            )
        else:  # single multi-turn
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_1=answer_1_a, answer_2=answer_2_a
            )
    else:
        # Single-turn:
        if '{question_2}' in template:
            raise ValueError('Template not compatible with single-turn')
        if judge_type == 'pair':
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer_a=answer_1_a,
                answer_b=answer_1_b
            )
        else:
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer=answer_1_a
            )

    # Call API
    try:
        response = client.chat.completions.create(
            model=judge_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            max_tokens=2048
        )
        judgement = response.choices[0].message.content.strip()
    except Exception as e:
        error_message = str(e)
        if "Request too large" in error_message and "413" in error_message:
            print(f"Request too large for q_id below -> verdict = 'unknown'")
            judgement = 'unknown'
        else:
            raise e # Re-raise exceptions other than 'Request too large'

    verdict_match = re.search(r'\[\[(A|B|C|\d+)\]\]', judgement)
    verdict = verdict_match.group(1) if verdict_match else "Unknown"

    conversation_a = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_a, "role": "assistant"},
        {"content": question_2, "role": "user"},
        {"content": answer_2_a, "role": "assistant"},
    ]
    conversation_b = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_b, "role": "assistant"},
        {"content": question_2, "role": "user"},
        {"content": answer_2_b, "role": "assistant"},
    ]

    new_row = {
        'question_id': row['question_id'],
        'category': row['category'],
        'model_a': model_a,
        'model_b': model_b,
        'conversation_a': conversation_a,
        'conversation_b': conversation_b,
        'judge_name': judge_name,
        'judge_type': judge_type,
        'judgement_explanation': judgement,
        'verdict': verdict
    }

    return new_row

In [ ]:
# Functions to Process and Save All Questions
def process_judgements(merged_df, output_path):
    existing_ids = set()
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        existing_ids = set(existing_df['question_id'].unique())
        print(f"Loaded {len(existing_ids)} existing judgements from {output_path}. Will skip these.")
        # Skip questions that have already been judged
        merged_df = merged_df[len(existing_ids):]

    judgements = []
    for _, row in merged_df.iterrows():
        try:
            q_id = row['question_id']
            if q_id in existing_ids:
                continue

            judgement = generate_judgement(groq_client, row, judge_model)
            judgements.append(judgement)

            print(f"Done question_id {q_id}.")
            time.sleep(10)  # Rate limit

        except Exception as e:
            print(f"Error on question_id {q_id}: {e}")
            print('End process!')
            break

    return judgements

In [ ]:
# Run
judgements = process_judgements(merged_df, output_file_path)

In [ ]:
df_judgements = pd.DataFrame(judgements)
df_judgements

In [ ]:
if os.path.exists(output_file_path):
    # If file exists, append without writing header
    df_judgements.to_csv(output_file_path, mode='a', header=False, index=False)
else:
    # If file does not exist, write with header
    df_judgements.to_csv(output_file_path, mode='w', header=True, index=False)

print(f"Data has been written to the file {output_file_path}")